# 3. Embedding Generation

This notebook generates deep visual embeddings for the complete
validated product catalog using the pretrained ResNet-50 feature
extractor developed in Notebook 2.

### Objectives

- Load the validated product metadata
- Recreate the ResNet-50 feature extractor
- Process product images in batches
- Generate a 2048-dimensional embedding for each product
- Preserve the mapping between embeddings and product IDs
- Monitor embedding generation progress
- Save the complete embedding matrix for later similarity search

The generated embeddings will be used by the similarity-search pipeline
in the next notebook.

## 1. Imports

The required libraries are imported for loading metadata, processing
images, running the ResNet-50 feature extractor, tracking progress,
and storing the generated embeddings.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

d:\Projects\VisualProductSearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Project Paths

This notebook uses the cleaned product metadata generated in Notebook 1
and the ResNet-50 model configuration established in Notebook 2.

Generated embeddings will be stored in the processed-data directory.

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_PATH = PROCESSED_DIR / "product_metadata.csv"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "resnet50_feature_extractor.pth"

EMBEDDINGS_PATH = PROJECT_ROOT / "embeddings" / "product_embeddings.npy"
EMBEDDING_IDS_PATH = PROJECT_ROOT / "embeddings" / "embedding_product_ids.npy"

print("Metadata:", METADATA_PATH)
print("Model:", MODEL_PATH)
print("Embeddings:", EMBEDDINGS_PATH)
print("Embedding IDs:", EMBEDDING_IDS_PATH)

Metadata: d:\Projects\VisualProductSearch\data\processed\product_metadata.csv
Model: d:\Projects\VisualProductSearch\models\resnet50_feature_extractor.pth
Embeddings: d:\Projects\VisualProductSearch\embeddings\product_embeddings.npy
Embedding IDs: d:\Projects\VisualProductSearch\embeddings\embedding_product_ids.npy


In [3]:
print("Metadata exists:", METADATA_PATH.exists())
print("Model exists:", MODEL_PATH.exists())

Metadata exists: True
Model exists: True


## 3. Load Product Metadata

The validated metadata generated in Notebook 1 is loaded to determine
which images must be processed and to preserve the product ordering
associated with each embedding.

In [4]:
metadata_df = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata_df.shape)
print("Number of products:", len(metadata_df))

Metadata shape: (44441, 11)
Number of products: 44441


In [5]:
missing_paths = metadata_df["image_path"].apply(
    lambda path: not Path(path).exists()
).sum()

print("Missing image paths:", missing_paths)

Missing image paths: 0


## 4. Configure Computation Device

Embedding generation involves running thousands of images through a
CNN, so GPU acceleration is important for reducing inference time.

The notebook automatically selects CUDA when available and otherwise
falls back to the CPU.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 2050


## 5. Image Preprocessing

Every product image must undergo the same preprocessing used when the
ResNet-50 feature extractor was validated in Notebook 2.

Using a consistent preprocessing pipeline ensures that the embeddings
are generated in the same feature space.

In [7]:
image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 6. Dataset for Embedding Generation

A PyTorch Dataset is used to load product images and apply the required
preprocessing.

The dataset returns both the processed image and its product ID.

The product ID is preserved so that every generated embedding can be
mapped back to the corresponding product.

In [8]:
class EmbeddingDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "id": int(row["id"])
        }

In [9]:
embedding_dataset = EmbeddingDataset(
    metadata_df,
    transform=image_transform
)

print("Dataset size:", len(embedding_dataset))

Dataset size: 44441


In [10]:
sample = embedding_dataset[0]

print("Image shape:", sample["image"].shape)
print("Product ID:", sample["id"])

Image shape: torch.Size([3, 224, 224])
Product ID: 15970


## 7. Configure Batch Inference

The complete catalog will be processed in batches rather than one
image at a time.

A batch size of 16 is used initially because the available GPU has
4 GB of VRAM. The batch size can be increased later if GPU memory
usage allows.

The DataLoader uses `shuffle=False` so that the order of product IDs
remains aligned with the order of generated embeddings.

In [11]:
BATCH_SIZE = 16

data_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(data_loader))

Batch size: 16
Number of batches: 2778


## 8. Load the ResNet-50 Feature Extractor

The ResNet-50 architecture is recreated and the feature-extractor
checkpoint saved in Notebook 2 is loaded.

The ImageNet classification layer is replaced with `nn.Identity()`,
matching the architecture used during Notebook 2 verification.

The model is then frozen and placed in evaluation mode because it is
being used only for inference.

In [12]:
resnet50 = models.resnet50(weights=None)

resnet50.fc = nn.Identity()

resnet50.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

resnet50 = resnet50.to(device)
resnet50.eval()

for parameter in resnet50.parameters():
    parameter.requires_grad = False

print("ResNet-50 feature extractor loaded.")

ResNet-50 feature extractor loaded.


## 9. Test Batch Feature Extraction

Before processing the complete catalog, a single batch is passed
through the feature extractor.

This verifies that:

- The DataLoader produces the expected tensor shape
- The model and input are on the same device
- The feature dimension is 2048
- Batch inference works correctly

In [13]:
first_batch = next(iter(data_loader))

images = first_batch["image"].to(device)

print("Input batch shape:", images.shape)

with torch.no_grad():
    batch_features = resnet50(images)

print("Feature batch shape:", batch_features.shape)

Input batch shape: torch.Size([16, 3, 224, 224])
Feature batch shape: torch.Size([16, 2048])


In [14]:
print("First batch IDs:")
print(first_batch["id"][:10])

First batch IDs:
tensor([15970, 39386, 59263, 21379, 53759,  1855, 30805, 26960, 29114, 30039])


## 10. Generate Embeddings for the Complete Catalog

The complete validated catalog is processed in batches.

For each batch:

1. Product images are loaded and preprocessed
2. Images are moved to the selected device
3. ResNet-50 generates a 2048-dimensional representation
4. The resulting features are moved back to CPU memory
5. Product IDs are stored in the same order

No gradients are calculated because this is an inference-only
operation.

The final result will contain one 2048-dimensional embedding for
each of the 44,441 products.

In [15]:
all_embeddings = []
all_ids = []

with torch.no_grad():

    for batch in tqdm(data_loader, desc="Generating embeddings"):

        images = batch["image"].to(device, non_blocking=True)

        features = resnet50(images)

        features = features.cpu().numpy()

        all_embeddings.append(features)
        all_ids.extend(batch["id"].numpy())

Generating embeddings: 100%|██████████| 2778/2778 [12:16<00:00,  3.77it/s]


In [16]:
embeddings = np.vstack(all_embeddings)
product_ids = np.array(all_ids)

print("Embeddings shape:", embeddings.shape)
print("Product IDs shape:", product_ids.shape)

Embeddings shape: (44441, 2048)
Product IDs shape: (44441,)


In [17]:
assert embeddings.shape == (len(metadata_df), 2048)
assert product_ids.shape == (len(metadata_df),)
assert len(np.unique(product_ids)) == len(product_ids)

print("✓ Number of embeddings:", len(embeddings))
print("✓ Embedding dimension:", embeddings.shape[1])
print("✓ Product IDs are unique")
print("✓ Embeddings and IDs are aligned")

✓ Number of embeddings: 44441
✓ Embedding dimension: 2048
✓ Product IDs are unique
✓ Embeddings and IDs are aligned


In [18]:
print("Embedding dtype:", embeddings.dtype)
print("Minimum value:", embeddings.min())
print("Maximum value:", embeddings.max())
print("Mean value:", embeddings.mean())
print("Standard deviation:", embeddings.std())

Embedding dtype: float32
Minimum value: 0.0
Maximum value: 10.601868
Mean value: 0.06266266
Standard deviation: 0.24635334


In [19]:
print("Contains NaN:", np.isnan(embeddings).any())
print("Contains Inf:", np.isinf(embeddings).any())

Contains NaN: False
Contains Inf: False


## 11. Save the Embedding Matrix

The generated embeddings are saved as a NumPy array.

A separate array containing the corresponding product IDs is also
saved.

Keeping these files separate allows the embedding matrix to remain
compact while preserving the mapping required to retrieve product
metadata later.

In [20]:
np.save(EMBEDDINGS_PATH, embeddings)
np.save(EMBEDDING_IDS_PATH, product_ids)

print("Embeddings saved to:", EMBEDDINGS_PATH)
print("Product IDs saved to:", EMBEDDING_IDS_PATH)

Embeddings saved to: d:\Projects\VisualProductSearch\embeddings\product_embeddings.npy
Product IDs saved to: d:\Projects\VisualProductSearch\embeddings\embedding_product_ids.npy


In [21]:
loaded_embeddings = np.load(EMBEDDINGS_PATH)
loaded_product_ids = np.load(EMBEDDING_IDS_PATH)

print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded IDs shape:", loaded_product_ids.shape)

Loaded embeddings shape: (44441, 2048)
Loaded IDs shape: (44441,)


In [22]:
assert np.array_equal(embeddings, loaded_embeddings)
assert np.array_equal(product_ids, loaded_product_ids)

print("✓ Saved embeddings verified")
print("✓ Saved product-ID mapping verified")

✓ Saved embeddings verified
✓ Saved product-ID mapping verified


# 12. Notebook Summary

This notebook generated deep visual embeddings for the complete
validated product catalog using the pretrained ResNet-50 feature
extractor developed in Notebook 2.

### Embedding Generation

- Products processed: **44,441**
- Model: **ResNet-50**
- Input size: **224 × 224**
- Embedding dimension: **2,048**
- Feature extraction mode: **Frozen**
- Gradient computation: **Disabled**
- Batch processing: **Enabled**
- DataLoader shuffling: **Disabled** to preserve embedding-product alignment

The resulting embedding matrix has the expected shape:

`44,441 × 2,048`

A separate product-ID array was generated to maintain the mapping between
each embedding and its corresponding catalog product.

### Validation

The generated embeddings were verified for:

- Correct number of embeddings
- Correct embedding dimension
- Unique product IDs
- Embedding-to-product alignment
- Absence of NaN values
- Absence of infinite values

The saved embedding files were also reloaded and compared against the
original arrays to verify successful persistence.

### Output Files

The generated files are:

`data/processed/product_embeddings.npy`

`data/processed/embedding_product_ids.npy`

### Next Step

The complete catalog now has a numerical visual representation that can
be searched.

In **Notebook 4**, cosine similarity and FAISS will be used to build
the visual similarity-search system and retrieve the top-K visually
similar products for a query image.